# Train/Test Split and Exploratory Data Analysis

## Introduction

This notebook prepares the cleaned welding dataset for the machine learning stage. The first objective is to create a train/test split while avoiding data leakage, especially when several rows may correspond to the same physical weld. After the split, the training set will be used for exploratory data analysis, including correlations, PCA, and feature selection. The test set will remain untouched until the final evaluation of the models.

In [1]:
import pandas as pd

df = pd.read_csv("../data/welddb_cleaned_units.csv")

df.head()

,C,Si,Mn,S,P,Ni,Cr,Mo,V,O_ppm,...,charpy_toughness_J,weld_id,weld_type_FCA,weld_type_GMAA,weld_type_GTAA,weld_type_MMA,weld_type_SA,weld_type_SAA,weld_type_TSA,AC_DC_unknown
0,0.037,0.30,0.65,0.008,0.012,0.0,0.0,0.0,0.0,423.0,...,NaN,Evans-Ni/CMn-1990/1991-0Aaw,0,0,0,1,0,0,0,0
1,0.037,0.30,0.65,0.008,0.012,0.0,0.0,0.0,0.0,423.0,...,100.0,Evans-Ni/CMn-1990/1991-0Aawch,0,0,0,1,0,0,0,0
2,0.037,0.30,0.65,0.008,0.012,0.0,0.0,0.0,0.0,423.0,...,100.0,Evans-Ni/CMn-1990/1991-0Aht,0,0,0,1,0,0,0,0
3,0.037,0.31,1.03,0.007,0.014,0.0,0.0,0.0,0.0,423.0,...,NaN,Evans-Ni/CMn-1990/1991-0Baw,0,0,0,1,0,0,0,0
4,0.037,0.31,1.03,0.007,0.014,0.0,0.0,0.0,0.0,423.0,...,100.0,Evans-Ni/CMn-1990/1991-0Bawch,0,0,0,1,0,0,0,0


## 2. Inspect Weld Identifiers

Before splitting the dataset, we need to check how the weld identifiers are structured. Some rows may represent different measurements of the same physical weld. These rows must stay together in either the training set or the test set to avoid data leakage.

In [2]:
print("Number of rows:", len(df))
print("Number of unique weld IDs:", df["weld_id"].nunique())

df["weld_id"].value_counts().head(30)

Number of rows: 1639
Number of unique weld IDs: 1477


weld_id
Pat-1981-RE2L/BX200         5
Pat-1981-S2/BX200           5
Pat-1981-RE3L/BX200         5
Pat-1981-WB3/BX200          5
Pat-1981-S4/BX200           5
Pat-1981-S6/BX200           5
Pat-1981-S2Mo/BX200         5
Pat-1981-S4Mo/BX200         5
Pat-1981-RE2L/BX400         5
Pat-1981-S2/BX400           5
Pat-1981-RE3L/BX400         5
Pat-1981-WB3/BX400          5
Pat-1981-S4/BX400           5
Pat-1981-S6/BX400           5
Pat-1981-S2Mo/BX400         5
Evans-B/9Cr1Mo-1994-CM57    4
Evans-B/9Cr1Mo-1994-CM58    4
Evans-B/9Cr1Mo-1994-CM59    4
Evans-B/9Cr1Mo-1994-CM60    4
Evans-B/9Cr1Mo-1994-CM61    4
Evans-B/9Cr1Mo-1994-CM62    4
Evans-B/9Cr1Mo-1994-CM63    4
Evans-B/9Cr1Mo-1994-CM64    4
Pat-1981-S4Mo/BX400         4
RC81033                     4
Evans-AlTi-1994-80/8ht      3
Evans-AlTi-1994-80/47ht     3
Evans-AlTi-1994-80/160ht    3
Evans-AlTi-1994-80/280ht    3
Evans-AlTi-1994-80/480ht    3
Name: count, dtype: int64

In [3]:
df[["weld_id"]].head(50)

,weld_id
0,Evans-Ni/CMn-1990/1991-0Aaw
1,Evans-Ni/CMn-1990/1991-0Aawch
2,Evans-Ni/CMn-1990/1991-0Aht
3,Evans-Ni/CMn-1990/1991-0Baw
4,Evans-Ni/CMn-1990/1991-0Bawch
5,Evans-Ni/CMn-1990/1991-0Bht
6,Evans-Ni/CMn-1990/1991-0Caw
7,Evans-Ni/CMn-1990/1991-0Cawch
8,Evans-Ni/CMn-1990/1991-0Cht
9,Evans-Ni/CMn-1990/1991-0Daw


### 2.1 Investigate related weld IDs

Some weld identifiers share the same base name but have different suffixes such as `aw`, `awch`, `ht`, or `mic`. Before creating the train/test split, we need to determine whether these rows correspond to the same underlying weld or to genuinely different weld conditions.

In [4]:
cols_to_compare = [
    "weld_id",
    "C", "Si", "Mn", "Ni", "Cr", "Mo",
    "current_A", "voltage_V",
    "heat_input_kJ_mm",
    "interpass_temp_C",
    "pwht_temp_C",
    "pwht_time_h"
]

df[df["weld_id"].str.contains("Evans-Ni/CMn-1990/1991-0A", na=False)][cols_to_compare]

,weld_id,C,Si,Mn,Ni,Cr,Mo,current_A,voltage_V,heat_input_kJ_mm,interpass_temp_C,pwht_temp_C,pwht_time_h
0,Evans-Ni/CMn-1990/1991-0Aaw,0.037,0.3,0.65,0.0,0.0,0.0,170.0,21.0,1.0,200.0,250.0,14.0
1,Evans-Ni/CMn-1990/1991-0Aawch,0.037,0.3,0.65,0.0,0.0,0.0,170.0,21.0,1.0,200.0,0.0,0.0
2,Evans-Ni/CMn-1990/1991-0Aht,0.037,0.3,0.65,0.0,0.0,0.0,170.0,21.0,1.0,200.0,580.0,2.0


In [5]:
examples = [
    "Evans-Ni/CMn-1990/1991-0A",
    "Pat-1981-WB3/BX200",
    "Evans-B/9Cr1Mo-1994-CM63"
]

for ex in examples:
    print("\n###", ex)
    display(df[df["weld_id"].str.contains(ex, na=False)][[
        "weld_id",
        "C", "Si", "Mn", "Ni", "Cr", "Mo",
        "current_A", "voltage_V",
        "heat_input_kJ_mm",
        "interpass_temp_C",
        "pwht_temp_C",
        "pwht_time_h"
    ]])


### Evans-Ni/CMn-1990/1991-0A


,weld_id,C,Si,Mn,Ni,Cr,Mo,current_A,voltage_V,heat_input_kJ_mm,interpass_temp_C,pwht_temp_C,pwht_time_h
0,Evans-Ni/CMn-1990/1991-0Aaw,0.037,0.3,0.65,0.0,0.0,0.0,170.0,21.0,1.0,200.0,250.0,14.0
1,Evans-Ni/CMn-1990/1991-0Aawch,0.037,0.3,0.65,0.0,0.0,0.0,170.0,21.0,1.0,200.0,0.0,0.0
2,Evans-Ni/CMn-1990/1991-0Aht,0.037,0.3,0.65,0.0,0.0,0.0,170.0,21.0,1.0,200.0,580.0,2.0



### Pat-1981-WB3/BX200


,weld_id,C,Si,Mn,Ni,Cr,Mo,current_A,voltage_V,heat_input_kJ_mm,interpass_temp_C,pwht_temp_C,pwht_time_h
960,Pat-1981-WB3/BX200,0.08,0.31,1.67,0.0,0.0,0.03,625.0,75.36,4.6,200.0,0.0,0.0
961,Pat-1981-WB3/BX200,0.08,0.31,1.67,0.0,0.0,0.03,625.0,75.36,4.6,200.0,0.0,0.0
962,Pat-1981-WB3/BX200,0.08,0.31,1.67,0.0,0.0,0.03,625.0,75.36,4.6,200.0,0.0,0.0
963,Pat-1981-WB3/BX200,0.08,0.31,1.67,0.0,0.0,0.03,625.0,75.36,4.6,200.0,0.0,0.0
964,Pat-1981-WB3/BX200,0.08,0.31,1.67,0.0,0.0,0.03,625.0,75.36,4.6,200.0,0.0,0.0



### Evans-B/9Cr1Mo-1994-CM63


,weld_id,C,Si,Mn,Ni,Cr,Mo,current_A,voltage_V,heat_input_kJ_mm,interpass_temp_C,pwht_temp_C,pwht_time_h
208,Evans-B/9Cr1Mo-1994-CM63,0.086,0.2,1.04,0.0,8.98,1.01,180.0,34.0,1.0,175.0,750.0,8.0
209,Evans-B/9Cr1Mo-1994-CM63,0.086,0.2,1.04,0.0,8.98,1.01,180.0,34.0,1.0,175.0,750.0,8.0
210,Evans-B/9Cr1Mo-1994-CM63,0.086,0.2,1.04,0.0,8.98,1.01,180.0,34.0,1.0,175.0,750.0,8.0
211,Evans-B/9Cr1Mo-1994-CM63,0.086,0.2,1.04,0.0,8.98,1.01,180.0,34.0,1.0,175.0,750.0,8.0


## 3. Train/Test Split

Grouping by `weld_id` is not enough to avoid data leakage. The same weld is often stored under several `weld_id`s: for example `Mart-A` holds the tensile test of a weld, and `Mart-Aa` to `Mart-Ak` hold the Charpy tests of the same weld at 11 temperatures. Split by `weld_id`, these rows end up on both sides, and the model is tested on a weld it has already seen at the neighbouring temperatures.

The examples above also show that the `aw`, `awch` and `ht` rows of a weld share the same composition and welding parameters, and only differ by the post-weld heat treatment.

Rows are therefore grouped by identical inputs: every input column except the post-weld heat treatment must be equal. We cannot be sure that such rows are literally the same weld, but the model only sees these columns, so for the model they are the same input. All rows of a group are kept in the same split.

In [6]:
from sklearn.model_selection import GroupShuffleSplit

targets = [
    "yield_strength_MPa",
    "uts_MPa",
    "elongation_pct",
    "reduction_area_pct",
    "charpy_temp_C",
    "charpy_toughness_J"
]

input_cols = [c for c in df.columns if c not in targets + ["weld_id", "pwht_temp_C", "pwht_time_h"]]
input_group = df.groupby(input_cols).ngroup()

print("Number of identical-input groups:", input_group.nunique())

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, groups=input_group)
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

Number of identical-input groups: 620
Train rows: 1332
Test rows: 307


### 3.1 Leakage Check

In [7]:
overlap = set(input_group.iloc[train_idx]) & set(input_group.iloc[test_idx])

print("Number of identical-input groups present in both train and test:", len(overlap))

overlap_ids = set(train_df["weld_id"]) & set(test_df["weld_id"])

print("Number of weld IDs present in both train and test:", len(overlap_ids))

mart_a = df["weld_id"].str.match(r"Mart-A[a-z]?$")
print("Mart-A rows in train:", mart_a.iloc[train_idx].sum(), "| in test:", mart_a.iloc[test_idx].sum())

Number of identical-input groups present in both train and test: 0
Number of weld IDs present in both train and test: 6
Mart-A rows in train: 12 | in test: 0


No identical-input group is present in both sets, and the 12 rows of `Mart-A` (1 tensile test and 11 Charpy tests) all stay on the same side.

6 `weld_id`s are still present in both sets: `RC81033` and five `Stil-TiB-1978` IDs. Their rows have the same name but a different composition (for example C goes from 0.029 to 0.100 across the 4 `RC81033` rows), so they are different welds that were given the same name in the database, not a leak.

In [8]:
print("Train percentage:", round(len(train_df) / len(df) * 100, 2), "%")
print("Test percentage:", round(len(test_df) / len(df) * 100, 2), "%")

print("Unique weld IDs in train:", train_df["weld_id"].nunique())
print("Unique weld IDs in test:", test_df["weld_id"].nunique())

Train percentage: 81.27 %
Test percentage: 18.73 %
Unique weld IDs in train: 1195
Unique weld IDs in test: 288


### 3.2 Save the Split

The split is saved once, so that the EDA and every model use the same training rows and are evaluated on the same test rows. The `input_group` column is saved with it: the cross-validation folds built later on `train.csv` must keep each group in a single fold, for the same reason as the train/test split.

In [9]:
train_df.assign(input_group=input_group.iloc[train_idx].values).to_csv("../data/train.csv", index=False)
test_df.assign(input_group=input_group.iloc[test_idx].values).to_csv("../data/test.csv", index=False)

print("Saved train:", train_df.shape, "| test:", test_df.shape)

Saved train: (1332, 37) | test: (307, 37)


## 4. Define Potential Target Variables

Before performing PCA or feature selection, we need to decide which variables represent weld quality. Mechanical properties such as yield strength, tensile strength, elongation, Charpy toughness, hardness, and related measurements are possible target variables. These variables should not be included as input features if they are used to define or predict weld quality.

In [10]:
train_df.columns.tolist()

['C',
 'Si',
 'Mn',
 'S',
 'P',
 'Ni',
 'Cr',
 'Mo',
 'V',
 'O_ppm',
 'Ti_ppm',
 'N_ppm',
 'Al_ppm',
 'Nb_ppm',
 'current_A',
 'voltage_V',
 'AC_DC',
 'electrode_polarity',
 'heat_input_kJ_mm',
 'interpass_temp_C',
 'pwht_temp_C',
 'pwht_time_h',
 'yield_strength_MPa',
 'uts_MPa',
 'elongation_pct',
 'reduction_area_pct',
 'charpy_temp_C',
 'charpy_toughness_J',
 'weld_id',
 'weld_type_FCA',
 'weld_type_GMAA',
 'weld_type_GTAA',
 'weld_type_MMA',
 'weld_type_SA',
 'weld_type_SAA',
 'weld_type_TSA',
 'AC_DC_unknown']

In [11]:
target_candidates = [
    "yield_strength_MPa",
    "uts_MPa",
    "elongation_pct",
    "reduction_area_pct",
    "charpy_temp_C",
    "charpy_toughness_J"
]

target_summary = pd.DataFrame({
    "non_missing": train_df[target_candidates].notna().sum(),
    "missing": train_df[target_candidates].isna().sum(),
    "missing_pct": train_df[target_candidates].isna().mean() * 100
})

target_summary

,non_missing,missing,missing_pct
yield_strength_MPa,620,712,53.453453
uts_MPa,596,736,55.255255
elongation_pct,558,774,58.108108
reduction_area_pct,559,773,58.033033
charpy_temp_C,723,609,45.720721
charpy_toughness_J,723,609,45.720721


## 5. Training Set Overview

Before selecting the final target and features, we examine the general structure of the training data. This includes dataset size, missing values, and the distribution of the main input variables.

In [12]:
print("Training set shape:", train_df.shape)

missing_train = pd.DataFrame({
    "missing_count": train_df.isna().sum(),
    "missing_pct": train_df.isna().mean() * 100
}).sort_values("missing_pct", ascending=False)

missing_train

Training set shape: (1332, 37)


,missing_count,missing_pct
elongation_pct,774,58.108108
reduction_area_pct,773,58.033033
uts_MPa,736,55.255255
yield_strength_MPa,712,53.453453
charpy_temp_C,609,45.720721
charpy_toughness_J,609,45.720721
P,0,0.000000
Mn,0,0.000000
Si,0,0.000000
C,0,0.000000


### Observation

All input features are complete in the training set after preprocessing. Missing values are only present in the mechanical-property variables that may be used as prediction targets. Therefore, no additional imputation is required for the input features at this stage.